# Week-6 Assignment

**Name:** Atharv Patil

**Dataset:** source.csv (10,000 Records, Synthetic)

In [ ]:
#Import Required Libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [ ]:
#Create Spark Session
spark= SparkSession.builder\
    .appName("Week6Assignment")\
    .getOrCreate()

### Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

**The Driver** is the main program of a Spark application. It creates the SparkSession, divides the work into tasks, and sends them to the executors. It also collects the final results.

**The Cluster** Manager is responsible for managing the cluster resources. It allocates CPU and memory to Spark applications and starts executors on worker nodes.

**Executors** are processes that run on worker nodes. They execute the tasks assigned by the Driver, process the data, and return the results. They can also store data in memory for faster processing.

### Q2. How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

Spark does not execute transformations immediately. Instead, it remembers all the operations and creates a plan. The actual execution starts only when an action like `show()`, `count()`, or `collect()` is called.

This improves performance because Spark can optimize the execution plan, remove unnecessary steps, and reduce data movement. As a result, large datasets are processed faster and more efficiently.

### Q3. Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

In [ ]:
df= spark.read\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .csv("data/source.csv")

df.show(5)

+--------+-------+----------+------------+---------------+----------+--------+---------+--------+---------+------+--------+-----------+----------+
|order_id|user_id|product_id|product_name|       category|base_price|   price|   amount|quantity|   status|region|priority|   old_name|order_date|
+--------+-------+----------+------------+---------------+----------+--------+---------+--------+---------+------+--------+-----------+----------+
|   10001| 5012.0|     P2679|       Mixer|Home Appliances|   2098.36| 2098.36|  7428.19|       3|Completed|  West|    High|  Product_8|2026-02-17|
|   10002| 4257.0|     P9928|       Table|      Furniture|  40477.89|40477.89| 47763.91|       1|  Shipped| South|  Medium|Product_151|2026-05-23|
|   10003| 5552.0|     P6514|       Phone|    Electronics|  55881.34|55881.34|197819.94|       3|Completed| North|  Medium| Product_25|2026-07-03|
|   10004| 9785.0|     P7201|      Jacket|       Clothing|  64589.55|64589.55|304862.68|       4|Completed|  East|    

### Q4. What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

CSV is a row-based file format, while Parquet is a column-based file format.

CSV stores data row by row, so Spark has to read the complete row even if only a few columns are needed. This makes processing slower.

Parquet stores data column by column. Spark reads only the required columns, which reduces disk I/O, uses less memory, and improves performance. It also provides better compression, so the file size is usually smaller.

### Q5. Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.

In [ ]:
df.select("product_id", "price")\
  .filter(col("category") == "Electronics")\
  .show(5)

+----------+--------+
|product_id|   price|
+----------+--------+
|     P6514|55881.34|
|     P3103|58579.42|
|     P1117| 1780.64|
|     P1887|52752.72|
|     P4006|15408.05|
+----------+--------+
only showing top 5 rows


### Q6. Write the code to revise a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

In [ ]:
df= df.withColumnRenamed("old_name", "new_name")\
       .withColumn("price", col("price").cast("double"))

df.printSchema()
df.show(5)

root
 |-- order_id: integer (nullable = true)
 |-- user_id: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- order_date: date (nullable = true)

+--------+-------+----------+------------+---------------+----------+--------+---------+--------+---------+------+--------+-----------+----------+
|order_id|user_id|product_id|product_name|       category|base_price|   price|   amount|quantity|   status|region|priority|   new_name|order_date|
+--------+-------+----------+------------+---------------+----------+--------+---------+--------+---------+------+--------+-----------+-----

### Q7. How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Spark keeps a Lineage Graph (DAG) that records all the transformations applied to the data. Instead of storing multiple copies of the data, Spark remembers how the data was created.

If a worker node fails and some data is lost, Spark uses the Lineage Graph to recreate only the missing data by running the required transformations again. This makes Spark fault tolerant and helps it recover without processing the entire dataset from the beginning.

### Q8. Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

In [ ]:
df_orders= df
df_orders.filter((col("status") == "Completed") & (col("amount") > 1000)).show(5)

+--------+-------+----------+------------+---------------+----------+--------+---------+--------+---------+------+--------+-----------+----------+
|order_id|user_id|product_id|product_name|       category|base_price|   price|   amount|quantity|   status|region|priority|   new_name|order_date|
+--------+-------+----------+------------+---------------+----------+--------+---------+--------+---------+------+--------+-----------+----------+
|   10001| 5012.0|     P2679|       Mixer|Home Appliances|   2098.36| 2098.36|  7428.19|       3|Completed|  West|    High|  Product_8|2026-02-17|
|   10003| 5552.0|     P6514|       Phone|    Electronics|  55881.34|55881.34|197819.94|       3|Completed| North|  Medium| Product_25|2026-07-03|
|   10004| 9785.0|     P7201|      Jacket|       Clothing|  64589.55|64589.55|304862.68|       4|Completed|  East|     Low|Product_159|2026-07-05|
|   10016| 4770.0|     P1117|      Tablet|    Electronics|   1780.64| 1780.64| 10505.78|       5|Completed| North|    

### Q9. Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

Predicate Pushdown is a feature of Parquet that allows Spark to apply filter conditions while reading the file. Instead of loading the complete dataset into memory, Spark reads only the rows that match the filter condition.

This reduces the amount of data loaded into memory, decreases disk I/O, and improves the overall performance. It is one of the main reasons why Parquet is preferred for big data processing.

### Q10. Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [ ]:
df= df.withColumn("final_price", col("base_price") * 1.18)
df.select("base_price", "final_price").show(5)

+----------+-----------------+
|base_price|      final_price|
+----------+-----------------+
|   2098.36|        2476.0648|
|  40477.89|       47763.9102|
|  55881.34|       65939.9812|
|  64589.55|        76215.669|
|  56395.29|66546.44219999999|
+----------+-----------------+
only showing top 5 rows


### Q11. What is the difference between Transformations and Actions? Provide two examples of each.

Transformations are operations that create a new DataFrame or RDD from an existing one. They are lazy, which means Spark does not execute them immediately. Instead, Spark waits until an action is called and then executes all the transformations together.

Actions are operations that trigger the execution of all previous transformations and return the final result or write the data to storage.

Examples of Transformations:
- `select()`
- `filter()`

Examples of Actions:
- `show()`
- `count()`

### Q12. Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [ ]:
#saving CSV DataFrame as Parquet

df.write\
    .mode("overwrite")\
    .parquet("parquet_data")

In [ ]:
spark.read.parquet("parquet_data")\
    .filter(col("user_id").isNotNull())\
    .write\
    .mode("overwrite")\
    .option("header", "true")\
    .csv("output.csv")

### Q13. In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

In Client Mode, the Driver program runs on the machine from where the Spark application is submitted. If that machine gets disconnected or stops working, the application may also stop.

In Cluster Mode, the Driver runs inside the cluster. Even if the user's machine disconnects, the application continues running because the Driver is managed by the cluster itself.

Client Mode is mostly used for development and testing, while Cluster Mode is preferred for production environments.

### Q14. Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

In [ ]:
df.filter((col("region") == "North") | (col("priority") == "High")).show(5)

+--------+-------+----------+------------+---------------+----------+--------+---------+--------+---------+------+--------+-----------+----------+------------------+
|order_id|user_id|product_id|product_name|       category|base_price|   price|   amount|quantity|   status|region|priority|   new_name|order_date|       final_price|
+--------+-------+----------+------------+---------------+----------+--------+---------+--------+---------+------+--------+-----------+----------+------------------+
|   10001| 5012.0|     P2679|       Mixer|Home Appliances|   2098.36| 2098.36|  7428.19|       3|Completed|  West|    High|  Product_8|2026-02-17|         2476.0648|
|   10003| 5552.0|     P6514|       Phone|    Electronics|  55881.34|55881.34|197819.94|       3|Completed| North|  Medium| Product_25|2026-07-03|        65939.9812|
|   10005| 4733.0|     P2307| Cricket Bat|         Sports|  56395.29|56395.29| 66546.44|       1|  Pending| North|  Medium| Product_72|2026-08-21| 66546.44219999999|
|   

### Q15. When exploring a dataset, why is it safer to use `.show(5)` instead of `.collect()` on a multi-terabyte dataset?

The `.show(5)` function displays only the first five rows of the dataset. It reads only a small amount of data, so it is fast and uses very little memory.

The `.collect()` function brings the entire dataset from all worker nodes to the Driver. If the dataset is very large, it can use a huge amount of memory and may even crash the application.

For large datasets, it is always safer to use `.show(5)` while exploring the data.

## Brief Insights on Performance and Architecture

During this assignment, I learned how Spark's architecture helps process large datasets efficiently. The Driver, Cluster Manager, and Executors work together to divide the workload and execute tasks in parallel. This allows Spark to make better use of cluster resources and complete jobs faster. Spark also uses a Lineage Graph (DAG), which helps recover lost data if a worker node fails without restarting the entire job.

I also understood how Spark improves performance using features like Lazy Evaluation, Predicate Pushdown, and columnar file formats such as Parquet. Instead of executing every operation immediately, Spark first creates an optimized execution plan. Reading only the required columns and filtering data while loading also reduces memory usage and speeds up processing. These features make Spark a powerful framework for handling big data efficiently.